# 01 Data Preprocessing and EDA


## Objective

???????????????????????????????


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import CLASSIFICATION_TARGET, FIGURES_DIR, RANDOM_STATE, RESULTS_DIR
from data_utils import (
    describe_dataframe,
    ensure_project_dirs,
    load_raw_dataset,
    missing_value_summary,
    save_processed_dataset,
    validate_required_columns,
)
from feature_engineering import add_behavior_features
from visualization import (
    plot_correlation_heatmap,
    plot_missingness,
    plot_numeric_histograms,
    plot_target_distribution,
)

np.random.seed(RANDOM_STATE)
ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\final_report_project


## Load, Validate, and Save Processed Dataset


In [2]:
df_raw = load_raw_dataset(download=True)
schema = validate_required_columns(df_raw)
df_processed = add_behavior_features(df_raw)
processed_path = save_processed_dataset(df_processed)
print(f"Processed dataset saved to: {processed_path}")
print(f"Processed shape: {df_processed.shape}")

display(schema)
df_processed.head()


Processed dataset saved to: C:\Users\qintian\Desktop\大数据\final_report_project\data\processed\digital_lifestyle_benchmark_2025_processed.csv
Processed shape: (3500, 31)


,column,present,dtype,missing_count
0,id,True,int64,0
1,age,True,int64,0
2,gender,True,object,0
3,region,True,object,0
4,income_level,True,object,0
5,education_level,True,object,0
6,daily_role,True,object,0
7,device_hours_per_day,True,float64,0
8,phone_unlocks,True,int64,0
9,notifications_per_day,True,int64,0


,id,age,gender,region,income_level,education_level,daily_role,device_hours_per_day,phone_unlocks,notifications_per_day,social_media_mins,study_mins,physical_activity_days,sleep_hours,sleep_quality,anxiety_score,depression_score,stress_level,happiness_score,focus_score,high_risk_flag,device_type,productivity_score,digital_dependence_score,social_media_hours,study_hours,notifications_per_device_hour,unlocks_per_device_hour,device_to_sleep_ratio,activity_sleep_interaction,social_to_study_ratio
0,1,40,Female,Asia,High,High School,Part-time/Shift,3.54,45,561,98,34,7.0,9.123800,3.353627,9.926651,5.0,6.593289,8.0,23.0,0,Android,70.000000,25.700000,1.633333,0.566667,158.474576,12.711864,0.387996,63.866600,2.800000
1,2,27,Male,Africa,Lower-Mid,Master,Full-time Employee,5.65,100,393,174,102,2.0,8.837517,2.908147,4.000000,4.0,4.126926,8.1,35.0,0,Laptop,64.000000,30.100000,2.900000,1.700000,69.557522,17.699115,0.639320,17.675034,1.689320
2,3,31,Male,North America,Lower-Mid,Bachelor,Full-time Employee,8.87,181,231,595,140,1.0,6.486743,2.889213,4.000000,8.0,1.429139,7.6,15.0,0,Android,65.299301,40.600000,9.916667,2.333333,26.042841,20.405862,1.367404,6.486743,4.219858
3,4,41,Female,Middle East,Low,Master,Caregiver/Home,4.05,94,268,18,121,4.0,7.600504,3.097488,7.093357,9.0,4.995512,7.8,28.0,1,Tablet,80.000000,36.684152,0.300000,2.016667,66.172840,23.209877,0.532859,30.402016,0.147541
4,5,26,Female,Europe,Lower-Mid,Bachelor,Full-time Employee,13.07,199,91,147,60,1.0,5.197962,2.786098,7.028125,15.0,9.448757,4.2,70.0,1,Android,65.299301,48.400000,2.450000,1.000000,6.962510,15.225708,2.514447,5.197962,2.409836


## Summary Tables

???????? CSV????????


In [3]:
descriptive_summary = describe_dataframe(df_processed)
missing_summary = missing_value_summary(df_processed)
target_distribution = (
    df_processed[CLASSIFICATION_TARGET]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis(CLASSIFICATION_TARGET)
    .reset_index(name="count")
)
target_distribution["ratio"] = target_distribution["count"] / target_distribution["count"].sum()

schema.to_csv(RESULTS_DIR / "preprocessing_schema_check.csv", index=False)
descriptive_summary.to_csv(RESULTS_DIR / "eda_descriptive_summary.csv", index=False)
missing_summary.to_csv(RESULTS_DIR / "missing_value_summary.csv", index=False)
target_distribution.to_csv(RESULTS_DIR / "eda_high_risk_flag_distribution.csv", index=False)

display(target_distribution)
display(missing_summary.head(10))


,high_risk_flag,count,ratio
0,0,2795,0.798571
1,1,705,0.201429


,column,missing_count,missing_rate
0,id,0,0.0
1,age,0,0.0
2,gender,0,0.0
3,region,0,0.0
4,income_level,0,0.0
5,education_level,0,0.0
6,daily_role,0,0.0
7,device_hours_per_day,0,0.0
8,phone_unlocks,0,0.0
9,notifications_per_day,0,0.0


## EDA Figures

????? `figures/` ???


In [4]:
numeric_columns = df_processed.select_dtypes(include=[np.number]).columns.tolist()
plot_target_distribution(df_processed, CLASSIFICATION_TARGET, FIGURES_DIR / "eda_high_risk_flag_distribution.png")
plot_missingness(missing_summary, FIGURES_DIR / "eda_missing_value_rate.png")
plot_numeric_histograms(df_processed, numeric_columns, FIGURES_DIR / "eda_numeric_histograms.png")
plot_correlation_heatmap(df_processed, FIGURES_DIR / "eda_numeric_correlation_heatmap.png")
print("Saved EDA figures to:")
for name in [
    "eda_high_risk_flag_distribution.png",
    "eda_missing_value_rate.png",
    "eda_numeric_histograms.png",
    "eda_numeric_correlation_heatmap.png",
]:
    print(FIGURES_DIR / name)


Saved EDA figures to:
C:\Users\qintian\Desktop\大数据\final_report_project\figures\eda_high_risk_flag_distribution.png
C:\Users\qintian\Desktop\大数据\final_report_project\figures\eda_missing_value_rate.png
C:\Users\qintian\Desktop\大数据\final_report_project\figures\eda_numeric_histograms.png
C:\Users\qintian\Desktop\大数据\final_report_project\figures\eda_numeric_correlation_heatmap.png


## Notes for Report

? notebook ??? EDA ????????????????????
